# Introduction to the Quant Research Project

## Overview

This project is a quantitative research framework for the **E‑mini S&P 500 (ES) futures market**. It combines:

- A **fully automated data pipeline** that collects, cleans, and stores 15‑minute candles.
- **Interactive research notebooks** for pattern analysis, game theory, and backtesting.
- A **validated intraday strategy** ready for prop firm accounts (no overnight positions)(Maybe Montecarlo or more advanced stuff needed for it to be ready for prod? idk).

All code, analysis, and results are reproducible and version‑controlled. The system is designed to be **maintained with minimal manual intervention**.

## Data Pipeline – Real‑Time Updates with GitHub Actions

### Data Source

Raw tick data was originally obtained from a public Kaggle dataset (historical ES futures). I processed those ticks into cleaned **15‑minute OHLCV bars** (open, high, low, close, volume) covering the period **2020‑02‑02 to 2026‑05‑07**.

### Database – Supabase

The cleaned data lives in a **Supabase PostgreSQL database** (cloud‑hosted). Using Supabase provides:

- A simple REST API for querying data from Jupyter notebooks.
- Built‑in authentication (RLS policies) so that researchers can read data but cannot modify it.
- Automatic backups and high availability.

### Automated Weekly Updates (GitHub Actions)

The market evolves, so the dataset must stay current. I built a **GitHub Actions workflow** that runs every Saturday at 4:00 AM UTC. It:

1. Connects to Yahoo Finance (`ES=F` ticker) and downloads the latest 15‑minute candles.
2. Deduplicates and validates the new data.
3. Pushes the new candles to the Supabase table `es_15min`.

This ensures that the research environment always reflects the most recent market activity **without any manual data fetching**.

## Quantitative Research Workflow

The analysis follows a structured, step‑by‑step approach (tasks 1‑9), each documented in its own notebook.

### Task 1 – Price Pattern Analysis

- Computed daily returns, best/worst days, day‑of‑week and month‑of‑year effects.
- Identified winning/losing streaks and typical correction sizes after large moves.
- Used **all available data (2020‑2026)** for statistical robustness.

### Task 2 – Volatility Analysis

- Calculated daily ranges and classified periods into low, normal, and high volatility.
- Studied what happens after extreme volatility days (mean‑reversion tendency).
- Confirmed volatility clustering (autocorrelation of squared returns).

### Task 3 – Game Theory & Market Participants

- Identified four main participants: institutions, momentum traders, mean‑reversion traders, retail.
- Defined three market scenarios: support/resistance test, after a strong move, range breakout.
- Built qualitative behavior tables for each participant under different market conditions.

### Task 4 – Payoff Matrices

- Created 2×2 payoff matrices for each scenario (your decision vs. market majority).
- Found Nash equilibria – pure equilibria exist only for range breakouts (follow majority).
- Concluded that acting **contrarian at support/resistance** yields positive expectancy.

### Task 5 – Trading Rules Design

- Derived concrete entry rules from the game theory insights.
- Chose a **single, simple rule**: **Fade support (long only)** when price opens within 0.1% of a validated swing low.
- Defined exit rules: take profit at +0.6× ATR, stop loss at –0.4× ATR.
- Added filters: RTH session only (9:30‑15:50 ET), ATR(14) > 5, level age > 1 hour.

### Task 6 – Automated Backtest

- Implemented a fully automated backtest engine (no manual data entry).
- Used **6.25 years of 15‑minute candles** (41,639 bars after RTH filter).
- Simulated 1 contract of ES, with forced position closure at 15:50 ET to comply with prop firm rules.
- **Results:** 6,406 trades, 49% win rate, profit factor 1.40, total return 255%, annualised return 22.47%, max drawdown –2.87%.

### Task 7 – Strengths & Weaknesses

- Analysed best and worst trades; identified that the strategy works best in high‑volatility regimes.
- Noted that win rate is only 49% – psychological resilience is required.
- Proposed improvements: add a trend filter, time‑based exit, and dynamic position sizing.

### Task 8 – Visualisations

- Created equity curve, drawdown chart, return distribution, monthly heatmap, and win rate by month.
- Generated real‑price charts (using Supabase data) for the best and worst trades.

### Task 9 – Personal Playbook

- Summarised the strategy in a **trader’s playbook** with a memorable metaphor (“The Trout Fisher”).
- Included a detailed checklist, risk management rules, and an adaptation plan for changing volatility.

## Final Remarks

This project demonstrates a **professional, production‑ready quant workflow**:

- Clean, version‑controlled data pipeline.
- Rigorous statistical and game‑theoretic analysis.
- An intraday strategy that respects prop firm rules (drawdown < 8%, no overnight positions).
- A systematic way to adapt the strategy to changing market conditions.

The strategy is not a “black box” – every rule is transparent and can be modified or extended.

**Thank you for exploring this work.** Feel free to use the code as a foundation for your own quant research or live trading.